# 1. Project Title: House Price Prediction with Machine Learning

**Kit:** AI Practical Project Learning Kit · **Project 1 of 5** · **Domain:** Machine Learning (Regression) · **Difficulty:** Beginner

Open this notebook in Google Colab and run the cells **from top to bottom** (`Runtime → Run all`, or press `Shift + Enter` on each cell). No GPU is needed.

## 2. Project Overview

In this project you teach a computer to **estimate the price of a house from facts about its neighbourhood** (income, house age, number of rooms, location and so on). You never write the pricing rule yourself. Instead you show the computer thousands of examples with the correct answer and let an algorithm *learn* the rule. That is **Machine Learning**.

You will build two models, a simple **Linear Regression** and a more powerful **Random Forest**, measure them honestly on houses they have never seen, compare them, and save the better one so that it can be reused in an app.

## 3. Real-World Problem Statement

A property website wants to show buyers a quick *estimated value* for any listing, and a bank wants a sanity check on loan applications. Hand-written rules ("add 10% for a garden") do not capture how many factors interact. A model trained on past sales can.

**Our task:** given the 8 numeric features of a California census block, predict the **median house value** of that block.

> **Important:** the data are 1990 census figures, so the predictions are in *1990-era dollars*. This is a **learning exercise**, not a tool for pricing real houses today.

## 4. Project Objectives

By the end you will be able to:

1. Explain **supervised learning** and **regression** in your own words.
2. Load a dataset into a pandas `DataFrame` and explore it.
3. Clean data and explain why the cleaning was needed.
4. Split data into **training** and **test** sets and explain data leakage.
5. Select useful features.
6. Train **Linear Regression** and **Random Forest** models.
7. Measure them with **MAE, MSE, RMSE and R²** and read each number correctly.
8. Compare models, predict for new inputs, and **save / load** a model with Joblib.

## 5. What We Will Build

A small prediction system:

`Input: 8 neighbourhood numbers  →  trained model  →  Output: predicted median house value`

with graphs for the data, the predictions, the errors and a comparison table of the two models.

## 6. Technologies and Libraries Used

| Library | What it does in this project |
|---|---|
| `numpy` | Fast numeric arrays and maths |
| `pandas` | Tables (`DataFrame`) for loading, cleaning and exploring data |
| `matplotlib`, `seaborn` | Drawing charts |
| `scikit-learn` | The dataset, train/test split, models and metrics |
| `joblib` | Saving and loading trained models |

All of them are pre-installed in Google Colab, so nothing needs to be downloaded or installed by hand.

## 7. System Architecture

```
California Housing data (scikit-learn)
        │
        ▼
 Load into DataFrame ─► Explore (EDA) ─► Clean
        │
        ▼
 Train / Test split (80% / 20%)
        │
        ▼
 Feature selection (fit on TRAIN only)
        │
        ├──► Linear Regression (with scaling)
        └──► Random Forest
                  │
                  ▼
   Evaluate on TEST: MAE · MSE · RMSE · R²
                  │
                  ▼
   Compare ─► Predict new input ─► Save best model (Joblib)
```

## 8. Understanding the Dataset

The **California Housing dataset** is built into scikit-learn and comes from the 1990 US census. **One row = one census block group** (a small neighbourhood of roughly 600–3,000 people), not one single house.

| Column | Meaning |
|---|---|
| `MedInc` | Median household income in the block (in tens of thousands of dollars) |
| `HouseAge` | Median age of houses in the block (years) |
| `AveRooms` | Average rooms per household |
| `AveBedrms` | Average bedrooms per household |
| `Population` | People living in the block |
| `AveOccup` | Average people per household |
| `Latitude`, `Longitude` | Location of the block |
| **`MedHouseVal`** *(target)* | Median house value, in units of **$100,000** (so `2.5` means $250,000) |

**Features** are the inputs (`X`); the **target** is the answer we want to predict (`y`). Because the target is a *number*, this is a **regression** problem. If it were a category such as "cheap / expensive", it would be *classification*.

## 9. Dataset Download / Loading

`fetch_california_housing` downloads the data the first time (about 1 MB) and caches it. `as_frame=True` gives us a pandas `DataFrame` directly.

In [ ]:
# Section 9 - imports and dataset loading
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42                     # makes results repeatable
np.random.seed(SEED)
sns.set_theme(style="whitegrid")

housing = fetch_california_housing(as_frame=True)
df = housing.frame            # features + target in one table
print("Rows, columns:", df.shape)
print("Target column:", housing.target_names)

## 10. Dataset Exploration (EDA)

**Exploratory Data Analysis** means looking before modelling. We check the size, data types, missing values and basic statistics.

In [ ]:
# Section 10a - first look at the table
display(df.head())
print()
df.info()
print()
print("Missing values per column:")
print(df.isnull().sum())

**How to read this:** `head()` shows the first five rows. `info()` lists each column's type and how many values are filled. `isnull().sum()` counts empty cells; zero everywhere means this dataset has no missing values (real datasets usually do, so you will often need `fillna` or `dropna`).

In [ ]:
# Section 10b - summary statistics
display(df.describe().T)

Look at the `max` of `MedHouseVal`. Many blocks sit exactly at the top value. We will deal with that in preprocessing. Also compare `AveRooms` (max in the dozens) with its median: a few blocks have extreme values, which are **outliers**.

In [ ]:
# Section 10c - distribution of the target and of two features
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df["MedHouseVal"], bins=50, ax=axes[0], color="steelblue")
axes[0].set_title("House value (target, x $100,000)")
sns.histplot(df["MedInc"], bins=50, ax=axes[1], color="seagreen")
axes[1].set_title("Median income (x $10,000)")
sns.histplot(df["HouseAge"], bins=40, ax=axes[2], color="darkorange")
axes[2].set_title("House age (years)")
plt.tight_layout(); plt.show()

**Reading the charts:** the value histogram is *right-skewed* (few expensive blocks) and has a **spike at 5.0** because the census capped values above $500,000. A model cannot learn the true price of capped houses, because the true price is hidden.

In [ ]:
# Section 10d - which features move together with price?
corr = df.corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation between columns (-1 to +1)")
plt.show()

print("Correlation with the target, strongest first:")
print(corr["MedHouseVal"].drop("MedHouseVal").sort_values(key=abs, ascending=False).round(3))

**Correlation** runs from −1 to +1. `MedInc` has the strongest link with price (richer areas have pricier houses). Correlation only captures *straight-line* relationships, so a feature with a low correlation (like `Latitude`) can still be useful in a non-linear model.

In [ ]:
# Section 10e - location matters: geographic scatter plot
plt.figure(figsize=(8, 6))
sc = plt.scatter(df["Longitude"], df["Latitude"], c=df["MedHouseVal"], s=df["Population"] / 100,
                 cmap="viridis", alpha=0.4)
plt.colorbar(sc, label="Median house value (x $100,000)")
plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.title("California blocks: colour = price, size = population")
plt.show()

The map is clearly shaped like California: expensive blocks cluster along the coast (Bay Area and Los Angeles). A straight-line model struggles with this, but a tree-based model can carve up the map into regions.

## 11. Data Preprocessing

Preprocessing turns raw data into a form a model can learn from. We do three things: **clean**, **split**, and **select features**.

In [ ]:
# Section 11a - cleaning: remove blocks whose price was capped at the maximum
cap = df["MedHouseVal"].max()
n_capped = int((df["MedHouseVal"] >= cap).sum())
print(f"Maximum value = {cap}; blocks at the cap = {n_capped} ({n_capped / len(df):.1%})")

# Capped targets are 'censored': the real price is higher but unknown.
# We drop them so the model is not trained on wrong answers. (Student challenge: try keeping them.)
df_clean = df[df["MedHouseVal"] < cap].copy()
df_clean = df_clean.drop_duplicates()
print("Rows before:", len(df), "| after cleaning:", len(df_clean))

**Why split the data?** If we tested the model on the same houses it learned from, it could just memorise them. So we hide **20 %** of the data (the *test set*) and only look at it at the end. `random_state=42` makes the shuffle repeatable so everybody gets the same split.

In [ ]:
# Section 11b - train / test split
FEATURES_ALL = [c for c in df_clean.columns if c != "MedHouseVal"]
X = df_clean[FEATURES_ALL]
y = df_clean["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

**Feature selection** keeps the most informative inputs. `SelectKBest(f_regression, k=6)` scores each feature by how strongly it is related to the target and keeps the 6 best.

> **Data leakage warning:** we fit the selector on the **training set only**. If we used the test set to choose features, information from the "hidden" data would leak into training and make our final score look better than it really is.

In [ ]:
# Section 11c - feature selection (fit on training data only)
selector = SelectKBest(score_func=f_regression, k=6).fit(X_train, y_train)
scores = pd.Series(selector.scores_, index=FEATURES_ALL).sort_values(ascending=False)
print("Feature scores (higher = more related to price):")
print(scores.round(1))

SELECTED = list(X_train.columns[selector.get_support()])
print("\nSelected features:", SELECTED)

X_train_s, X_test_s = X_train[SELECTED], X_test[SELECTED]

## 12. Understanding the Algorithm

### Linear Regression
Draws the best straight-line (in many dimensions) through the data:
`price = w1×income + w2×age + ... + b`. Training finds the weights `w` and the intercept `b` that make the total squared error smallest. It is fast and easy to explain, but it cannot capture curves or interactions (such as "location × income").

We put a `StandardScaler` in front. It rescales every feature to mean 0 and spread 1 so that large-number columns (Population) do not dominate small-number ones. Linear Regression's *predictions* do not change with scaling, but scaled weights are comparable and many other algorithms need it.

### Random Forest Regressor
A forest is many **decision trees**. A single tree asks a chain of yes/no questions ("income > 5? latitude < 34?") and ends in a price. One tree overfits easily, so the forest trains many trees, each on a random sample of rows and features, and **averages** their answers. Averaging cancels out individual mistakes.

* **estimator** = one tree; `n_estimators=100` means 100 trees (more trees: steadier results but slower; returns diminish after a few hundred).
* `random_state=42` fixes the randomness so a re-run gives the same forest.
* `n_jobs=-1` uses all CPU cores.

## 13. Building the Model

Here we only *create* the two models. Nothing is learned until we call `.fit()` in the next section.

In [ ]:
# Section 13 - create the models
lin_model = make_pipeline(StandardScaler(), LinearRegression())
rf_model = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)
print(lin_model)
print(rf_model)

## 14. Training the Model

`fit(X, y)` is where learning happens: the model sees the training features and the correct prices and adjusts itself. The Random Forest may take 20–60 seconds in Colab.

In [ ]:
# Section 14 - train both models on the TRAINING data only
import time
t0 = time.time(); lin_model.fit(X_train_s, y_train); t_lin = time.time() - t0
t0 = time.time(); rf_model.fit(X_train_s, y_train);  t_rf = time.time() - t0
print(f"Linear Regression trained in {t_lin:.2f}s")
print(f"Random Forest trained in    {t_rf:.2f}s")

## 15. Model Evaluation

We now predict the **unseen test set** and compare with the true prices using four metrics (all in units of $100,000):

| Metric | Meaning | Good value |
|---|---|---|
| **MAE** | Average absolute error, easy to interpret | Lower |
| **MSE** | Average *squared* error; punishes big mistakes | Lower |
| **RMSE** | √MSE, back in the original units | Lower |
| **R²** | Share of the price variation the model explains (1 = perfect, 0 = no better than guessing the average) | Higher |

In [ ]:
# Section 15 - evaluate on the TEST data
def evaluate(name, model):
    pred = model.predict(X_test_s)
    mse = mean_squared_error(y_test, pred)
    return pred, {"Model": name,
                  "MAE": mean_absolute_error(y_test, pred),
                  "MSE": mse,
                  "RMSE": np.sqrt(mse),
                  "R2": r2_score(y_test, pred)}

pred_lin, m_lin = evaluate("Linear Regression", lin_model)
pred_rf,  m_rf  = evaluate("Random Forest", rf_model)

results = pd.DataFrame([m_lin, m_rf]).set_index("Model")
display(results.round(4))

# Is the model overfitting? compare training score with test score
print("R2 on training data:  Linear =", round(lin_model.score(X_train_s, y_train), 4),
      "| Forest =", round(rf_model.score(X_train_s, y_train), 4))
print("R2 on test data:      Linear =", round(m_lin["R2"], 4), "| Forest =", round(m_rf["R2"], 4))

**Reading the result:** multiply MAE by $100,000 to get the typical error in dollars. If a model's training R² is much higher than its test R², it has partly *memorised* the training houses (**overfitting**). Random forests usually show a bigger gap than linear models, yet often still win on the test set. Your exact numbers are whatever the cell above printed; they are not quoted here because they come from your own run.

## 16. Data Visualization

Charts that matter for regression: **actual vs predicted**, **residuals** (errors), **model comparison** and **feature importance**.

In [ ]:
# Section 16a - actual vs predicted
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
for ax, pred, name in zip(axes, [pred_lin, pred_rf], ["Linear Regression", "Random Forest"]):
    ax.scatter(y_test, pred, s=8, alpha=0.3)
    lim = [0, max(y_test.max(), pred.max())]
    ax.plot(lim, lim, "r--", label="perfect prediction")
    ax.set_title(name); ax.set_xlabel("Actual value (x $100,000)"); ax.set_ylabel("Predicted value")
    ax.legend()
plt.tight_layout(); plt.show()

Points on the red dashed line are perfect predictions. The tighter the cloud around the line, the better the model.

In [ ]:
# Section 16b - residuals (actual - predicted)
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(y_test - pred_lin, bins=60, color="tomato", alpha=0.5, label="Linear", ax=ax)
sns.histplot(y_test - pred_rf, bins=60, color="steelblue", alpha=0.5, label="Random Forest", ax=ax)
ax.axvline(0, color="black"); ax.set_xlabel("Error = actual - predicted (x $100,000)")
ax.set_title("Distribution of prediction errors"); ax.legend(); plt.show()

In [ ]:
# Section 16c - model comparison and feature importance
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
results[["MAE", "RMSE"]].plot(kind="bar", ax=axes[0], rot=0, color=["#4c72b0", "#dd8452"])
axes[0].set_title("Error (lower is better)")
results[["R2"]].plot(kind="bar", ax=axes[1], rot=0, color="seagreen", legend=False)
axes[1].set_title("R2 (higher is better)"); axes[1].set_ylim(0, 1)

imp = pd.Series(rf_model.feature_importances_, index=SELECTED).sort_values()
imp.plot(kind="barh", ax=axes[2], color="purple")
axes[2].set_title("Random Forest feature importance")
plt.tight_layout(); plt.show()

**Feature importance** tells which inputs the forest relied on most. Compare it with the correlation heatmap from Section 10: importance can highlight features, such as location, that plain correlation under-rates.

## 17. Testing with New Inputs

A model is only useful if it can predict for houses it has never seen. We pass a new row with the **same columns in the same order** as training.

In [ ]:
# Section 17a - predict for a made-up neighbourhood
new_block = pd.DataFrame([{
    "MedInc": 5.0, "HouseAge": 20, "AveRooms": 6.0, "AveBedrms": 1.0,
    "Population": 1200, "AveOccup": 3.0, "Latitude": 34.05, "Longitude": -118.25}])   # near Los Angeles

new_block = new_block[FEATURES_ALL][SELECTED]     # keep only the selected features, same order
for name, mdl in [("Linear Regression", lin_model), ("Random Forest", rf_model)]:
    p = float(mdl.predict(new_block)[0])
    print(f"{name:18s}: {p:.3f} x $100,000  =  ${p * 100000:,.0f}  (1990 dollars)")

Now an **interactive example**. The helper asks for each number, accepts a blank answer to use the typical value, and rejects text. Type your own values in the box that appears under the cell.

In [ ]:
# Section 17b - interactive prediction with input validation
limits = {c: (float(df_clean[c].min()), float(df_clean[c].max()), float(df_clean[c].median())) for c in FEATURES_ALL}

def ask_float(name):
    lo, hi, med = limits[name]
    while True:
        try:
            raw = input(f"{name} (range {lo:.2f} to {hi:.2f}, Enter = {med:.2f}): ").strip()
        except EOFError:                 # no keyboard available (e.g. automated run)
            return med
        if raw == "":
            return med
        try:
            v = float(raw)
        except ValueError:
            print("  Please type a number."); continue
        if not (lo <= v <= hi):
            print(f"  Outside the range seen in training ({lo:.2f} to {hi:.2f}); predictions there are unreliable. Try again.")
            continue
        return v

row = {c: ask_float(c) for c in SELECTED}
user_df = pd.DataFrame([row])[SELECTED]
print("\nYour input:", row)
print(f"Random Forest estimate: ${float(rf_model.predict(user_df)[0]) * 100000:,.0f} (1990 dollars)")

## 18. Saving the Trained Model

**Joblib** saves Python objects (such as trained models) to a file so you can load them later without retraining. We save the model together with its feature list, so whoever loads it knows which columns to supply.

In [ ]:
# Section 18 - save, reload and verify
best_name = results["R2"].idxmax()
best_model = rf_model if best_name == "Random Forest" else lin_model
print("Best model on the test set:", best_name)

joblib.dump({"model": best_model, "features": SELECTED, "name": best_name}, "house_price_model.joblib", compress=3)

loaded = joblib.load("house_price_model.joblib")
same = np.allclose(loaded["model"].predict(X_test_s[loaded["features"]]), best_model.predict(X_test_s))
print("Reloaded model gives identical predictions:", same)

import os
print("File size: %.1f MB" % (os.path.getsize("house_price_model.joblib") / 1e6))
try:
    from google.colab import files        # only exists inside Colab
    files.download("house_price_model.joblib")
except ImportError:
    print("(Not running in Colab - the file is in the current folder.)")

## 19. Final Output

A summary of everything the project produced.

In [ ]:
# Section 19 - final summary
print("=" * 60)
print("HOUSE PRICE PREDICTION - FINAL SUMMARY")
print("=" * 60)
print(f"Rows used after cleaning : {len(df_clean)}")
print(f"Features used            : {SELECTED}")
display(results.round(4))
print(f"Best model               : {best_name}")
print(f"Typical error (MAE)      : about ${results.loc[best_name, 'MAE'] * 100000:,.0f} (1990 dollars)")

## 20. How the Complete Project Works

1. **Load** 20,640 census blocks with 8 features and a price.
2. **Explore** distributions, correlations and the map to understand the data and spot the price cap.
3. **Clean** by removing capped (censored) targets and duplicates.
4. **Split** into training and test sets so evaluation is honest.
5. **Select** the 6 most related features, using the training set only.
6. **Train** Linear Regression (with scaling) and a 100-tree Random Forest.
7. **Evaluate** both on the unseen test set with MAE, MSE, RMSE and R².
8. **Predict** for new inputs, with validation of user entries.
9. **Save** the best model with Joblib so an app can load it later.

## 21. Limitations

* The data are from **1990** and describe whole **neighbourhoods**, not individual houses.
* Capped prices were removed, so the model is weak for the most expensive areas.
* Nothing in the data captures condition, size of the plot or school quality.
* The model can be unfair or wrong for areas unlike those in training; inputs outside the training range are unreliable.
* This is a teaching project and must not be used to value property or decide loans.

## 22. Real-World Applications

* Property portals' automatic valuation estimates.
* Insurance and bank risk checks.
* City planning and affordability studies.
* The same regression workflow predicts sales, demand, energy use, delivery time and many other numbers.

## 23. Future Enhancements

* Try **Gradient Boosting** or **XGBoost**, and tune with `GridSearchCV`.
* Use **cross-validation** (`cross_val_score`) for a steadier estimate than one split.
* Add engineered features such as rooms per person.
* Build a small web app with **Streamlit** or **Gradio** that loads `house_price_model.joblib`.
* Move to a newer, richer dataset (for example Ames Housing).

## 24. Student Mini Challenges

1. Change `test_size` to 0.3 and `n_estimators` to 20 and 300. How do the metrics and training time change?
2. Set `k=8` (use all features). Does the score improve? Why or why not?
3. Keep the capped rows (skip the filter in Section 11a). What happens to the errors?
4. Add a third model, `GradientBoostingRegressor`, to the comparison table.
5. Create a feature `RoomsPerPerson = AveRooms / AveOccup`, retrain, and check whether it helps.
6. Use `cross_val_score(rf_model, X_train_s, y_train, cv=5, scoring="r2")` and report the mean and spread.
7. Write a function `predict_price(**features)` that loads the Joblib file and returns a dollar string.